# Step 1: Fetch FAERS Data via openFDA API

FAERS (FDA Adverse Event Reporting System) gives us:
- Drug name + role (suspect / concomitant)
- Reaction as **MedDRA PT name** (ground truth for fine-tuning)
- Seriousness flags (Death, Hospitalization, Life-threatening, Disability) — ground truth
- Dechallenge (`drugadditional`) and rechallenge (`drugrecurrence`) signals
- Patient demographics (age, sex)

All 10,000 records have no narrative text — narratives are generated by Qwen3-32B in notebook 03.

**Outputs:**
- `data/faers_raw.json` — structured records with `who_umc_preliminary` pre-computed from FAERS fields
- WHO-UMC is computed deterministically here (no LLM) using outcome + dechallenge/rechallenge signals

In [1]:
%pip install requests tqdm pandas -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
import json
import time
import os
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_FILE = DATA_DIR / "faers_raw.json"

FAERS_URL = "https://api.fda.gov/drug/event.json"

# How many total records to fetch
# 5000 gives a good dataset size; increase to 10000 if time allows
TARGET_RECORDS = 5000
BATCH_SIZE = 100   # max 1000 per request, but 100 is safer for rate limits
SLEEP_BETWEEN = 0.5  # seconds between requests (be a good API citizen)

In [ ]:
def fetch_faers_batch(skip: int, limit: int = BATCH_SIZE, max_retries: int = 4) -> list | None:
    """Fetch one batch of FAERS records.

    Returns:
        list of records  — success (may be empty)
        None             — 404, genuinely no more results, caller should stop
    Retries on timeout / connection errors with exponential backoff.
    If all retries fail, returns [] so the main loop advances skip and continues.
    """
    params = {
        "search": "_exists_:patient.reaction AND _exists_:patient.drug",
        "limit": limit,
        "skip": skip,
    }
    for attempt in range(max_retries):
        try:
            r = requests.get(FAERS_URL, params=params, timeout=60)
            if r.status_code == 200:
                return r.json().get("results", [])
            elif r.status_code == 404:
                return None  # sentinel: no more data at this skip
            elif r.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"  Rate limited at skip={skip}. Waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"  HTTP {r.status_code} at skip={skip}: {r.text[:120]}")
                time.sleep(2 ** attempt)
        except requests.Timeout:
            wait = 5 * (2 ** attempt)  # 5s → 10s → 20s → 40s
            print(f"  Timeout at skip={skip} (attempt {attempt + 1}/{max_retries}). Retrying in {wait}s...")
            time.sleep(wait)
        except requests.ConnectionError as e:
            wait = 5 * (2 ** attempt)
            print(f"  Connection error at skip={skip} (attempt {attempt + 1}/{max_retries}). Retrying in {wait}s...")
            time.sleep(wait)
        except requests.RequestException as e:
            print(f"  Unrecoverable request error at skip={skip}: {e}")
            return []

    print(f"  All {max_retries} retries exhausted at skip={skip}. Skipping batch and continuing.")
    return []


# ── WHO-UMC causality computed from structured FAERS fields ───────────────────
# drugadditional: "1"=withdrawn, "2"=dose reduced, "3"=unchanged, "4"=unknown
# drugrecurrence: "1"=reaction recurred on rechallenge, "2"=did not recur
# reaction_outcome: already mapped to human-readable string below
def compute_who_umc(outcome: str, serious: bool, seriousness_criteria: list,
                    drug_additional: str, drug_recurrence: str) -> str:
    """
    Deterministic WHO-UMC from FAERS structured fields.
    No LLM needed — avoids circular generation from synthetic narratives.
    """
    rechallenge_pos  = drug_recurrence  == "1"
    dechallenge_pos  = drug_additional in ("1", "2")  # withdrawn or dose reduced
    resolved         = outcome in ("Recovered/Resolved", "Recovering/Resolving")
    fatal            = outcome == "Fatal"

    if rechallenge_pos and dechallenge_pos:
        return "Certain"
    elif dechallenge_pos and resolved:
        return "Probable/Likely"
    elif resolved and serious:
        # Serious event that resolved — plausible drug causality even without dechallenge data
        return "Probable/Likely"
    elif resolved:
        return "Possible"
    elif fatal:
        return "Possible"
    elif outcome == "Not Recovered":
        return "Unlikely"
    else:
        return "Unassessable"


def extract_clean_record(raw: dict) -> dict | None:
    """Extract only the fields we need from a raw FAERS record.
    Returns None if the record lacks required fields.
    """
    patient = raw.get("patient", {})
    reactions = patient.get("reaction", [])
    drugs = patient.get("drug", [])

    if not reactions or not drugs:
        return None

    reaction_pt = reactions[0].get("reactionmeddrapt", "").strip()
    reaction_outcome_code = reactions[0].get("reactionoutcome", "")
    if not reaction_pt:
        return None

    suspect_drug = next(
        (d for d in drugs if d.get("drugcharacterization") == "1"),
        drugs[0]
    )
    drug_name = suspect_drug.get("medicinalproduct", "").strip()
    if not drug_name:
        return None

    serious = raw.get("serious", "2") == "1"
    criteria = []
    seriousness_map = {
        "seriousnessdeath":          "Death",
        "seriousnesslifethreatening":"Life-threatening",
        "seriousnesshospitalization":"Hospitalization",
        "seriousnessdisabling":      "Disability/Incapacity",
        "seriousnesscongeni":        "Congenital Anomaly",
        "seriousnessother":          "Medically Significant",
    }
    for key, label in seriousness_map.items():
        if raw.get(key) == "1":
            criteria.append(label)

    outcome_map = {
        "1": "Recovered/Resolved", "2": "Recovering/Resolving",
        "3": "Not Recovered", "4": "Fatal", "5": "Unknown", "6": "Unknown",
    }
    outcome = outcome_map.get(str(reaction_outcome_code), "Unknown")

    # Dechallenge / rechallenge fields — present in raw FAERS, used for WHO-UMC
    drug_additional = str(suspect_drug.get("drugadditional", ""))   # "1"=withdrawn
    drug_recurrence = str(suspect_drug.get("drugrecurrence", ""))   # "1"=recurred

    narrative   = raw.get("narrativeincludeclinical", "").strip()
    age         = patient.get("patientonsetage", "")
    sex_code    = str(patient.get("patientsex", ""))
    sex         = {"1": "male", "2": "female"}.get(sex_code, "unknown")
    dose        = suspect_drug.get("drugdosagetext", "")
    indication  = suspect_drug.get("drugindication", "")

    # Count concomitant drugs (used as alternative-cause signal)
    concomitant_count = sum(1 for d in drugs if d.get("drugcharacterization") == "2")

    return {
        "id":                   raw.get("safetyreportid", ""),
        "drug_name":            drug_name,
        "drug_dose":            dose,
        "drug_indication":      indication,
        "reaction_pt":          reaction_pt,
        "reaction_outcome":     outcome,
        "serious":              serious,
        "seriousness_criteria": criteria,
        "patient_age":          age,
        "patient_sex":          sex,
        "narrative":            narrative,
        "has_narrative":        bool(narrative),
        # Causality signals (may be empty string if not reported)
        "drug_additional":      drug_additional,
        "drug_recurrence":      drug_recurrence,
        "concomitant_drugs":    concomitant_count,
        # Pre-computed WHO-UMC (deterministic, no LLM)
        "who_umc_preliminary":  compute_who_umc(
            outcome, serious, criteria, drug_additional, drug_recurrence
        ),
    }


print("Functions defined. Ready to fetch.")

In [8]:
# Check if we already have data (resume support)
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE) as f:
        existing = json.load(f)
    print(f"Found existing file with {len(existing)} records. Delete data/faers_raw.json to re-fetch.")
else:
    existing = []
    print("No existing data. Starting fresh fetch.")

Found existing file with 900 records. Delete data/faers_raw.json to re-fetch.


In [ ]:
# ── Main fetch loop ──────────────────────────────────────────────────────────
if len(existing) >= TARGET_RECORDS:
    print(f"Already have {len(existing)} records. Skipping fetch.")
    records = existing
else:
    records = list(existing)
    skip = len(records)
    consecutive_failures = 0
    MAX_CONSECUTIVE_FAILURES = 5   # stop if 5 batches in a row fail (network down)

    print(f"Fetching {TARGET_RECORDS - len(records)} more records (starting at skip={skip})...")

    with tqdm(total=TARGET_RECORDS - len(records), unit="rec") as pbar:
        while len(records) < TARGET_RECORDS:
            batch = fetch_faers_batch(skip=skip)

            if batch is None:
                # 404 — API says no more results exist at this skip
                print(f"Reached end of available data at skip={skip}.")
                break

            if len(batch) == 0:
                # Transient failure (all retries exhausted) — advance skip and keep going
                consecutive_failures += 1
                skip += BATCH_SIZE
                if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                    print(f"{MAX_CONSECUTIVE_FAILURES} consecutive batch failures. Stopping.")
                    break
                continue

            consecutive_failures = 0  # reset on any success

            for raw in batch:
                clean = extract_clean_record(raw)
                if clean:
                    records.append(clean)
                    pbar.update(1)
                if len(records) >= TARGET_RECORDS:
                    break

            skip += len(batch)
            time.sleep(SLEEP_BETWEEN)

            # Checkpoint every 500 records
            if len(records) % 500 < BATCH_SIZE:
                with open(OUTPUT_FILE, "w") as f:
                    json.dump(records, f)
                pbar.set_postfix({"saved": len(records)})

    # Final save
    with open(OUTPUT_FILE, "w") as f:
        json.dump(records, f, indent=2)
    print(f"\nSaved {len(records)} records to {OUTPUT_FILE}")

In [ ]:
# ── Backfill: enrich existing records with new fields ─────────────────────────
# Run this if you have records fetched before the drug_additional / who_umc_preliminary
# fields were added. No API calls needed — computes from fields already in the file.

with open(OUTPUT_FILE) as f:
    records = json.load(f)

needs_backfill = "who_umc_preliminary" not in records[0] if records else False

if needs_backfill:
    print(f"Backfilling {len(records)} records with WHO-UMC and causality fields...")

    # Redefine compute_who_umc inline in case kernel was restarted without running cell-3
    def _compute_who_umc(outcome, serious, criteria, drug_additional="", drug_recurrence=""):
        rechallenge_pos = drug_recurrence == "1"
        dechallenge_pos = drug_additional in ("1", "2")
        resolved        = outcome in ("Recovered/Resolved", "Recovering/Resolving")
        fatal           = outcome == "Fatal"
        if rechallenge_pos and dechallenge_pos:
            return "Certain"
        elif dechallenge_pos and resolved:
            return "Probable/Likely"
        elif resolved and serious:
            return "Probable/Likely"
        elif resolved:
            return "Possible"
        elif fatal:
            return "Possible"
        elif outcome == "Not Recovered":
            return "Unlikely"
        else:
            return "Unassessable"

    for rec in records:
        # These fields are absent in old records — default to empty (unknown)
        rec.setdefault("drug_additional",  "")
        rec.setdefault("drug_recurrence",  "")
        rec.setdefault("concomitant_drugs", 0)
        rec["who_umc_preliminary"] = _compute_who_umc(
            rec.get("reaction_outcome", "Unknown"),
            rec.get("serious", False),
            rec.get("seriousness_criteria", []),
            rec.get("drug_additional", ""),
            rec.get("drug_recurrence", ""),
        )

    with open(OUTPUT_FILE, "w") as f:
        json.dump(records, f, indent=2)
    print(f"Saved enriched records to {OUTPUT_FILE}")

    from collections import Counter
    dist = Counter(r["who_umc_preliminary"] for r in records)
    print("\nWHO-UMC distribution (from outcome field):")
    for cat, count in sorted(dist.items(), key=lambda x: -x[1]):
        print(f"  {cat:<20}: {count:>5} ({count/len(records):.1%})")
else:
    print(f"Records already have who_umc_preliminary. No backfill needed. ({len(records)} records)")

In [ ]:
import pandas as pd

df = pd.DataFrame(records)
print(f"Total records: {len(df)}")
print(f"Records WITH narrative: {df['has_narrative'].sum()} ({df['has_narrative'].mean():.1%})")
print(f"Serious cases: {df['serious'].sum()} ({df['serious'].mean():.1%})")
print(f"\nTop 10 reactions (MedDRA PT):")
print(df['reaction_pt'].value_counts().head(10).to_string())
print(f"\nTop 10 drugs:")
print(df['drug_name'].value_counts().head(10).to_string())
print(f"\nSeriousness criteria distribution:")
from collections import Counter
crit_counts = Counter(c for row in df['seriousness_criteria'] for c in row)
for k, v in crit_counts.most_common():
    print(f"  {k}: {v}")

## Done
Data saved to `data/faers_raw.json`. Proceed to `02_explore_meddra.ipynb`.